In [1]:
!unzip "pakiet(1).zip" -d lab5

Archive:  pakiet(1).zip
replace lab5/__MACOSX/._pakiet? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [2]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch
import pickle
import os
import random


PAD_VALUE = 4999

class MusicAcordsDataset(Dataset):
    def __init__(self, data, has_targets=True, use_embedding=True):
        super().__init__()
        self.sequences = []
        self.classes = []
        self.has_targets = has_targets
        self.use_embedding = use_embedding

        for sample in data:
            if self.has_targets:
                seq = sample[0]
                self.classes.append(torch.tensor(sample[1], dtype=torch.long))
            else:
                seq = sample

            if self.use_embedding:
                self.sequences.append(torch.tensor(seq, dtype=torch.long)) # [seq_length]
            else:
                self.sequences.append(torch.tensor(seq, dtype=torch.float32).unsqueeze(1)) # [seq_length, 1]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, indx):
        if self.has_targets:
            return self.sequences[indx], self.classes[indx]
        return self.sequences[indx], None # For test dataset

def pad_collate(batch):
    xx = [item[0] for item in batch]
    yy = [item[1] for item in batch if item[1] is not None]

    x_lens = torch.tensor([len(x) for x in xx], dtype=torch.long)
    xx_pad = pad_sequence(xx, batch_first=True, padding_value=PAD_VALUE)

    if len(yy) > 0:
        yy_stack = torch.stack(yy)
        return xx_pad, yy_stack, x_lens

    return xx_pad, None, x_lens # for test dataset


def create_dataset(batch_size, use_embedding=True):
    with open(os.path.join("lab5", "pakiet", "test_no_target.pkl"), 'rb') as f:
        test_data = pickle.load(f)

    with open(os.path.join("lab5", "pakiet", "train.pkl"), 'rb') as f:
        data = pickle.load(f)

    random.seed(42)
    random.shuffle(data)

    train_size = int(0.7 * len(data))
    train_data = data[:train_size]
    val_data = data[train_size:]

    train_set = MusicAcordsDataset(train_data, has_targets=True, use_embedding=use_embedding)
    val_set = MusicAcordsDataset(val_data, has_targets=True, use_embedding=use_embedding)
    test_set = MusicAcordsDataset(test_data, has_targets=False, use_embedding=use_embedding)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, drop_last=False, collate_fn=pad_collate)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, drop_last=False, collate_fn=pad_collate)

    return train_loader, val_loader, test_loader

In [3]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch


class LSTM_Seq_Regressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bidirectional):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        dropout_prob = 0.5 if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            bidirectional = bidirectional,
            dropout = dropout_prob
        )

    def forward(self, x):
        all_outputs, (h_n, c_n) = self.lstm(x)
        return all_outputs, (h_n, c_n)


class MusicAuthorClassifier(nn.Module):
    def __init__(self, seq_model: LSTM_Seq_Regressor, num_classes, hidden_size, use_mean_pooling, bidirectional, use_embedding, vocab_size, embedding_dim, pad_value):
        super().__init__()
        self.use_mean_pooling = use_mean_pooling
        self.seq_model = seq_model
        self.bidirectional = bidirectional
        self.use_embedding = use_embedding

        if self.use_embedding:
            self.embedding = nn.Embedding(
                num_embeddings=vocab_size,
                embedding_dim=embedding_dim,
                padding_idx=pad_value
            )

        input_size = seq_model.hidden_size * 2 if bidirectional else seq_model.hidden_size

        self.classification_head = nn.Sequential(
            nn.BatchNorm1d(input_size),
            nn.Dropout(p=0.5),
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.BatchNorm1d(hidden_size),
            nn.Dropout(p=0.5),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x, x_len):
        if self.use_embedding:
            x = self.embedding(x)

        x_packed = pack_padded_sequence(x, x_len.cpu(), batch_first=True, enforce_sorted=False)

        all_outputs, (h_n, c_n) = self.seq_model(x_packed)

        if self.use_mean_pooling:
            unpacked_out, _ = pad_packed_sequence(all_outputs, batch_first=True) # [batch, seq, hidden_size]

            sum_hidden = torch.sum(unpacked_out, dim=1)
            pooled_hidden = sum_hidden / x_len.unsqueeze(1).float().to(sum_hidden.device)

            out = self.classification_head(pooled_hidden)

        elif self.bidirectional:
            forward_hidden = h_n[-2]
            backward_hidden = h_n[-1]
            last_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)   # [batch, seq, 2*hidden_size]
            out = self.classification_head(last_hidden)

        else:
            last_hidden = h_n[-1] # [batch, seq, hidden_size]
            out = self.classification_head(last_hidden)

        return out


def create_model(lstm_hidden_size, lstm_num_layers, lstm_is_bidirectional, classifier_hidden_size, classifier_use_mean_pooling, use_embedding, vocab_size, embedding_dim, pad_value):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    lstm_input_size = embedding_dim if use_embedding else 1

    lstm_base = LSTM_Seq_Regressor(
        input_size=lstm_input_size,
        hidden_size=lstm_hidden_size,
        num_layers=lstm_num_layers,
        bidirectional=lstm_is_bidirectional
    )

    model = MusicAuthorClassifier(
        seq_model=lstm_base,
        num_classes=5,
        hidden_size=classifier_hidden_size,
        use_mean_pooling=classifier_use_mean_pooling,
        bidirectional=lstm_is_bidirectional,
        use_embedding=use_embedding,
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        pad_value=pad_value
    ).to(device)

    return model

In [4]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch
import os


def train(epochs, model, train_loader, val_loader):
    loss_func = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    best_val_loss = float('inf')
    save_path = 'best_model.pth'

    for epoch in range(epochs):
        # TRAINING
        model.train()
        train_epoch_loss = 0.0

        for x, targets, x_len in train_loader:
            x, targets = x.to(device), targets.to(device)

            optimizer.zero_grad()

            preds = model(x, x_len)

            loss = loss_func(preds, targets)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_epoch_loss += loss.item()

        # VALIDATION
        model.eval()

        all_preds = []
        all_targets = []
        val_loss_total = 0.0

        for x, targets, x_len in val_loader:
            x, targets = x.to(device), targets.to(device)

            with torch.no_grad():
                logits = model(x, x_len)

                predicted_classes = torch.argmax(logits, dim=1)

                all_preds.append(predicted_classes)
                all_targets.append(targets)
                loss = loss_func(logits, targets)
                val_loss_total += loss.item()

        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)

        avg_train_loss = train_epoch_loss / len(train_loader)
        avg_val_loss = val_loss_total / len(val_loader)
        val_acc = torch.sum(all_preds == all_targets).float() / len(all_preds)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            saved_info = f"| The best model saved"
        else:
            saved_info = ""

        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch: {epoch:3d} | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}{saved_info}")

In [5]:
import torch
import pandas as pd
import os

def sample(model, test_loader, weights_path='best_model.pth'):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print(f"Pomyślnie załadowano wagi z pliku: {weights_path}")
    else:
        raise ValueError("Weights path not found")

    model.eval()
    all_preds = []

    with torch.no_grad():
        for x, _, x_len in test_loader:
            x = x.to(device)
            logits = model(x, x_len)
            predicted_classes = torch.argmax(logits, dim=1)
            all_preds.extend(predicted_classes.cpu().numpy())

    df = pd.DataFrame(data=all_preds, columns=["predicted_class"])
    df.to_csv("pred.csv", index=False)
    print("Preds saved")

In [6]:
import pickle
import os

with open(os.path.join("lab5", "pakiet", "train.pkl"), 'rb') as f:
    data = pickle.load(f)

max_index = 0
for sample in data:
    seq = sample[0]
    local_max = max(seq)
    if local_max > max_index:
        max_index = local_max

print(f"Największy indeks akordu w zbiorze to: {max_index}")

Największy indeks akordu w zbiorze to: 191.0


In [7]:
EPOCHS = 100
BATCH_SIZE = 32
LSTM_HIDDEN_SIZE = 64
CLASSIFIER_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 2
MEAN_POOLING = True
BIDIRECTIONAL = True

USE_EMBEDDING = True
EMBEDDING_DIM = 32
VOCAB_SIZE = 5000


print(f"Experiment: {EPOCHS=}, {BATCH_SIZE=}, {LSTM_HIDDEN_SIZE=}, {CLASSIFIER_HIDDEN_SIZE=}, {LSTM_NUM_LAYERS=}, {MEAN_POOLING=}, {BIDIRECTIONAL=}, {USE_EMBEDDING=}")

train_loader, val_loader, test_loader = create_dataset(BATCH_SIZE, USE_EMBEDDING)

model = create_model(
    lstm_hidden_size=LSTM_HIDDEN_SIZE,
    lstm_num_layers=LSTM_NUM_LAYERS,
    lstm_is_bidirectional=BIDIRECTIONAL,
    classifier_hidden_size=CLASSIFIER_HIDDEN_SIZE,
    classifier_use_mean_pooling=MEAN_POOLING,
    use_embedding=USE_EMBEDDING,
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    pad_value=PAD_VALUE
)

print("Training started")
train(EPOCHS, model, train_loader, val_loader)

print("Sampling started")
sample(model, test_loader)

Experiment: EPOCHS=100, BATCH_SIZE=32, LSTM_HIDDEN_SIZE=64, CLASSIFIER_HIDDEN_SIZE=128, LSTM_NUM_LAYERS=2, MEAN_POOLING=True, BIDIRECTIONAL=True, USE_EMBEDDING=True
Training started
